# Lab 01: Explore Lambda Functions & Mock Data

All infrastructure (Lambda, DynamoDB, SNS, Secrets Manager) was deployed by CloudFormation.
In this lab you will:
- Invoke the Lambda functions directly using built-in **mock data**
- Understand the event format and responses
- Inspect DynamoDB records created by the scan

> No DigiCert account needed — mock mode is the default.

> **Estimated time:** 15 minutes

## Load workshop config

In [ ]:
# Config is written by the SageMaker lifecycle script from SSM at space startup.
# If this fails, re-launch the JupyterLab space to trigger the lifecycle script.
import json, pathlib, boto3

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)

REPO_DIR = pathlib.Path(REPO_DIR)
lm  = boto3.client('lambda',        region_name=AWS_REGION)
ddb = boto3.resource('dynamodb',    region_name=AWS_REGION)
sm  = boto3.client('secretsmanager',region_name=AWS_REGION)

PRIORITY_EMOJI = {'EXPIRED': '💀', 'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡', 'LOW': '🟢'}

def invoke(fn, payload):
    r = lm.invoke(FunctionName=fn, InvocationType='RequestResponse',
                  Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    if 'FunctionError' in r:
        raise RuntimeError(raw.get('errorMessage', raw))
    return raw.get('body', raw)

print(f'Region  : {AWS_REGION}')
print(f'Table   : {CERT_TABLE_NAME}')
print(f'Lambda  : {LAMBDA_SCAN}')
print('✅ Environment ready')

## Scan for expiring certificates

`certagent-scan-certificates` queries DigiCert (or mock data) and writes results to DynamoDB.
Pass `use_mock: true` to use the built-in mock dataset.

In [ ]:
from tabulate import tabulate

result = invoke(LAMBDA_SCAN, {'use_mock': True, 'threshold_days': 30})
certs = result.get('certificates', [])

print(f'Found {len(certs)} certificate(s) expiring within 30 days:')
print()
rows = [[PRIORITY_EMOJI.get(c['priority'],''), c['priority'], c['common_name'],
         c['days_remaining'], c['valid_till'], c['order_id']] for c in certs]
print(tabulate(rows, headers=['', 'Priority', 'Domain', 'Days Left', 'Expires', 'Order'], tablefmt='github'))

## Try different threshold values

In [ ]:
for t in [7, 14, 30, 60]:
    r = invoke(LAMBDA_SCAN, {'use_mock': True, 'threshold_days': t})
    print(f'  Threshold {t:>2}d -> {r.get("total_expiring", 0)} certs')

## Verify DynamoDB was populated

In [ ]:
table = ddb.Table(CERT_TABLE_NAME)
items = table.scan()['Items']
print(f'DynamoDB items: {len(items)}')
for i in sorted(items, key=lambda x: int(x.get('days_remaining', 9999))):
    print(f'  {i["domain"]:<25} {i["priority"]:<10} status={i.get("renewal_status","?")}')

## List inventory Lambda

In [ ]:
result = invoke(LAMBDA_INVENTORY, {'status': 'all'})
summary = result.get('summary', {})
print('Inventory summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

## Lab 01 Complete

You explored the Lambda functions, saw how mock data works, verified DynamoDB writes,
and checked the inventory.

**Next:** `02_certificate_lifecycle.ipynb`